# Notebook 16: Model Fingerprinting via Embeddings

## Why This Matters
Given a set of embedding vectors, can you determine which model produced them — without access to the model itself? This matters for IP protection (was my model copied?), supply chain audits (which model is this API actually using?), and competitive intelligence (is this company using a model similar to mine?).

The key insight: different embedding models impose different geometric structures on the same input set. These structures are fingerprints. This is the `whodunit` foundation.

## Structure
1. Why embedding geometry is model-specific (narrative)
2. Building a reference geometry from known models
3. Geometric features: isotropy, intrinsic dimension, neighbor structure
4. Classifier-based fingerprinting
5. Black-box API fingerprinting
6. Bridge to label noise detection

## What You'll Learn
- How different models impose different geometric structures on the same inputs
- What isotropy, intrinsic dimensionality, and neighborhood overlap measure
- How to classify unknown embeddings by model identity
- The limits of fingerprinting (when models are too similar to distinguish)


## Background

### Why geometry is a fingerprint

Each embedding model maps inputs to a different region and structure of embedding space. Two models trained with different objectives on different data will produce:
- Different centroids (mean embedding of a corpus)
- Different anisotropy (some dimensions are used more than others)
- Different intrinsic dimensionality (how many effective dimensions carry information)
- Different neighborhood structure (which inputs are near each other)

These geometric properties are consistent across inputs — they're properties of the model, not individual embeddings. This makes them fingerprints.

### Isotropy

A perfectly isotropic embedding space uses all dimensions equally — embeddings are spread uniformly across the hypersphere. In practice, most embedding models are anisotropic: some dimensions carry far more variance. The degree of anisotropy (measured by the spread of singular values of the embedding matrix) is model-specific.

### Intrinsic dimensionality

Even in a 384-dimensional space, the embeddings may lie on a lower-dimensional manifold. The intrinsic dimension (estimated via participation ratio or two-NN estimator) tells you how many effective dimensions are being used. Models trained on different data or with different objectives tend to use different intrinsic dimensions.


In [ ]:
# %pip install -q sentence-transformers torch numpy scikit-learn matplotlib

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

# Load multiple models to compare
MODEL_NAMES = {
    "minilm":  "all-MiniLM-L6-v2",
    "mpnet":   "all-mpnet-base-v2",
    "e5":      "intfloat/e5-small-v2",
}

print("Loading embedding models...")
models = {}
for key, name in MODEL_NAMES.items():
    models[key] = SentenceTransformer(name, device=device)
    dim = models[key].get_sentence_embedding_dimension()
    print(f"  {key}: {name} ({dim}d)")

In [ ]:
# A probe corpus — same inputs to all models
probe_texts = [
    "machine learning models learn from data", "neural networks compute weighted sums",
    "transformers use self-attention mechanisms", "embeddings represent meaning as vectors",
    "the cat sat on the mat", "a quick brown fox jumps", "she sells seashells",
    "the stock market fell today", "interest rates affect bond prices",
    "quantum computers use qubits", "protein folding determines function",
    "climate change causes sea level rise", "renewable energy reduces emissions",
    "the painting was auctioned for millions", "ballet requires strength and flexibility",
    "the recipe calls for two cups of flour", "bake at 350 degrees for thirty minutes",
] * 10  # repeat for statistical stability

# Encode with all models
embeddings_by_model = {}
for key, model in models.items():
    embs = model.encode(probe_texts, show_progress_bar=False, batch_size=64)
    embeddings_by_model[key] = embs
    print(f"  {key}: {embs.shape}")

## 1. Geometric Fingerprint Features

In [ ]:
def compute_geometric_features(embeddings: np.ndarray) -> dict:
    """Compute model-specific geometric properties of an embedding set."""
    N, D = embeddings.shape

    # Normalize to unit sphere
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embs_normed = embeddings / (norms + 1e-8)

    # 1. Mean cosine similarity (isotropy proxy: lower = more isotropic)
    sample = embs_normed[:100]
    cos_sims = (sample @ sample.T).flatten()
    np.fill_diagonal((sample @ sample.T), np.nan)  # exclude self-sim
    mean_cos = float(np.nanmean(cos_sims))

    # 2. Singular value spectrum (anisotropy)
    svd = TruncatedSVD(n_components=min(50, D-1))
    svd.fit(embeddings)
    sv = svd.singular_values_
    sv_norm = sv / sv.sum()
    # Participation ratio = 1 / sum(sv_norm^2) — higher = more isotropic
    participation_ratio = 1.0 / float(np.sum(sv_norm**2))
    top5_concentration = float(sv_norm[:5].sum())  # how much variance in top 5 SVs

    # 3. Intrinsic dimension (two-NN estimator)
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=3).fit(embeddings[:200])
    dists, _ = nn.kneighbors(embeddings[:200])
    mu = dists[:, 2] / (dists[:, 1] + 1e-8)
    intrinsic_dim = float(1.0 / np.mean(np.log(mu + 1e-8)))

    # 4. Mean pairwise L2 distance
    sample2 = embeddings[:100]
    l2_dists = np.sqrt(((sample2[:, None] - sample2[None, :])**2).sum(axis=-1))
    np.fill_diagonal(l2_dists, np.nan)
    mean_l2 = float(np.nanmean(l2_dists))

    return {
        "mean_cosine_sim": mean_cos,
        "participation_ratio": participation_ratio,
        "top5_sv_concentration": top5_concentration,
        "intrinsic_dim_estimate": intrinsic_dim,
        "mean_l2_distance": mean_l2,
        "embedding_norm_mean": float(norms.mean()),
        "embedding_norm_std": float(norms.std()),
    }


print("Geometric fingerprint features per model:")
print()
features_by_model = {}
for key, embs in embeddings_by_model.items():
    feats = compute_geometric_features(embs)
    features_by_model[key] = feats
    print(f"  {key} ({MODEL_NAMES[key]}):")
    for fname, val in feats.items():
        print(f"    {fname:<30}: {val:.4f}")
    print()

## 2. Classifier-Based Fingerprinting

In [ ]:
# Given an embedding, predict which model produced it
# Build a training set: embeddings labeled by model
all_embs, all_labels = [], []
label_map = {key: i for i, key in enumerate(embeddings_by_model)}

for key, embs in embeddings_by_model.items():
    all_embs.append(embs)
    all_labels.extend([label_map[key]] * len(embs))

X = np.vstack(all_embs)
y = np.array(all_labels)

# Simple linear probe — if geometry differs, a linear classifier should work
from sklearn.model_selection import cross_val_score
sc = StandardScaler()
X_scaled = sc.fit_transform(X)

clf = LogisticRegression(max_iter=500, C=1.0, random_state=42)
cv_scores = cross_val_score(clf, X_scaled, y, cv=5, scoring='accuracy')

print(f"Cross-validated model identification accuracy:")
print(f"  {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print()
print(f"Random baseline: {1/len(label_map):.3f}")
print()
print("If accuracy >> baseline, the models have distinguishable geometric fingerprints.")
print("If accuracy ≈ baseline, the models are too similar to distinguish from embeddings alone.")

## 3. Black-Box API Fingerprinting (Narrative)

In practice, fingerprinting a production API you don't control is harder — you can only observe outputs, not intermediate representations. The approach:

1. **Probe set selection:** choose inputs that maximally discriminate between candidate models — inputs where model outputs diverge
2. **Geometric signature extraction:** compute isotropy, participation ratio, intrinsic dim on probe outputs
3. **Nearest-neighbor matching:** compare geometric signature to a database of known model signatures
4. **Uncertainty quantification:** some pairs of models are too similar to distinguish confidently

The main challenge: fine-tuned models share most of their geometry with the base model. Distinguishing `LLaMA-3-70B-base` from `LLaMA-3-70B-instruct` from their embeddings is very difficult. Distinguishing `all-MiniLM-L6-v2` from `text-embedding-ada-002` is straightforward.

This is the `whodunit` tool concept — practical for distinguishing architecturally different models, limited for fine-tuned variants.


## 4. Bridge to Label Noise Detection

Model fingerprinting uses the geometric structure of an embedding space to make inferences about the model that produced it. Label noise detection uses the same geometric structure to make inferences about the *data* — specifically, which samples have labels that conflict with where they sit in embedding space.

If a sample labeled "cat" sits close to all the "dog" samples in CLIP embedding space, that's evidence it might be mislabeled. Notebook 17 builds this intuition into a practical mislabel detection pipeline.


## Summary

| Feature | What it captures | Model specificity |
|---------|-----------------|------------------|
| Mean cosine similarity | Isotropy (uniform vs concentrated embeddings) | High |
| Participation ratio | How many dimensions are actively used | High |
| Intrinsic dimensionality | Effective manifold dimension | High |
| Top-5 SV concentration | Variance in leading directions | High |
| Cross-model classification | Combined discriminability | Definitive |

**Next:** Notebook 17 — Label Noise Detection. Using CLIP/DINOv2 embeddings and clustering to find mislabeled training examples — the `label-noise` foundation.
